# 0.5 Bandwidth Content: Authorization Lists And Blob Hashes

This notebook builds the block-level bandwidth content table after calldata and BAL estimation already exist.

As in EIP-8131, it adds the transaction-content components that are not covered by calldata + BAL:

- EIP-7702 authorization tuples, fetched from JSON-RPC type-4 transactions.
- EIP-4844 blob versioned hashes, counted from Xatu `execution_transaction.blob_hashes`.

For this notebook, we only produce bandwidth-facing data. Other resource dimensions are intentionally left for later notebooks.

In [1]:
import os
from pathlib import Path

import clickhouse_connect
import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from sim.rpc_authorizations import fetch_authorization_data_for_blocks
from sim.xatu_calldata import query_xatu_calldata_by_block

pd.options.display.float_format = "{:,.4f}".format

load_dotenv(PROJECT_ROOT / ".env")
missing = [name for name in ["CLICKHOUSE_USER", "CLICKHOUSE_PASSWORD"] if not os.environ.get(name)]
if missing:
    raise RuntimeError("Missing .env values: " + ", ".join(missing))

CLICKHOUSE_RAW_HOST = os.environ.get("CLICKHOUSE_RAW_HOST", "clickhouse-raw.xatu.ethpandaops.io")
raw_client = clickhouse_connect.get_client(
    host=CLICKHOUSE_RAW_HOST,
    port=443,
    secure=True,
    username=os.environ["CLICKHOUSE_USER"],
    password=os.environ["CLICKHOUSE_PASSWORD"],
)

ETHNODEOPS_API_KEY = os.environ.get("ETHNODEOPS_API_KEY") or os.environ.get("hoodi_api_key")
ETHNODEOPS_RPC = os.environ.get("ETHNODEOPS_RPC", "https://erigon.mainnet.rpc.ethnodeops.xyz")
ALCHEMY_RPC = os.environ.get("ALCHEMY_RPC")

if ETHNODEOPS_API_KEY:
    RPC_URL = ETHNODEOPS_RPC
    RPC_HEADERS = {"X-API-Key": ETHNODEOPS_API_KEY}
    RPC_PROVIDER_LABEL = "ethnodeops_erigon_mainnet" if "erigon." in RPC_URL else "ethnodeops_mainnet"
elif ALCHEMY_RPC:
    RPC_URL = ALCHEMY_RPC
    RPC_HEADERS = None
    RPC_PROVIDER_LABEL = "alchemy_mainnet"
else:
    raise RuntimeError("Missing ETHNODEOPS_API_KEY or ALCHEMY_RPC in .env")

print("raw", CLICKHOUSE_RAW_HOST, raw_client.query("SELECT version()").result_rows)
print("rpc_provider", RPC_PROVIDER_LABEL)

raw clickhouse-raw.xatu.ethpandaops.io [('26.2.5.45',)]
rpc_provider ethnodeops_erigon_mainnet


## Parameters

In [2]:
NETWORK = "mainnet"
START_BLOCK = 24_120_001
N_BLOCKS = 50
BLOCKS = list(range(START_BLOCK, START_BLOCK + N_BLOCKS))

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

AUTH_RECORDS_CSV = DATA_DIR / f"rpc_authorization_records_{min(BLOCKS)}_{max(BLOCKS)}.csv"
AUTH_SUMMARY_CSV = DATA_DIR / f"rpc_authorization_summary_{min(BLOCKS)}_{max(BLOCKS)}.csv"
CALLDATA_CSV = DATA_DIR / f"xatu_calldata_{min(BLOCKS)}_{max(BLOCKS)}.csv"
BAL_CSV = DATA_DIR / f"rpc_bal_summary_{min(BLOCKS)}_{max(BLOCKS)}.csv"
CONTENT_CSV = DATA_DIR / f"bandwidth_content_8131_{min(BLOCKS)}_{max(BLOCKS)}.csv"

WRITE_CSV = True
min(BLOCKS), max(BLOCKS), len(BLOCKS)

(24120001, 24120050, 50)

## Pull Authorization Lists

The helper first queries Xatu for type-4 transaction hashes, then fetches each raw transaction from RPC and decodes the full `authorizationList`. Bandwidth counts every authorization tuple, including invalid or duplicate tuples, because they are bytes on the wire.

For EIP-8131 floor accounting, each authorization tuple contributes `108 bytes * 64 gas/byte`.

In [3]:
auth_records, auth_summary = fetch_authorization_data_for_blocks(
    raw_client=raw_client,
    rpc_url=RPC_URL,
    block_numbers=BLOCKS,
    network=NETWORK,
    rpc_headers=RPC_HEADERS,
)

auth_summary["authorization_tuple_gas"] = (
    auth_summary["authorization_tuple_8131_bytes"] * 64
)

if WRITE_CSV:
    auth_records.to_csv(AUTH_RECORDS_CSV, index=False)
    auth_summary.to_csv(AUTH_SUMMARY_CSV, index=False)
    print(AUTH_RECORDS_CSV)
    print(AUTH_SUMMARY_CSV)

auth_summary.head()

/Users/william/PycharmProjects/eip-7999-research/data/rpc_authorization_records_24120001_24120050.csv
/Users/william/PycharmProjects/eip-7999-research/data/rpc_authorization_summary_24120001_24120050.csv


,block_number,type4_tx_count,authorization_tuple_count,authorization_tuple_rlp_bytes,authorization_tuple_8131_bytes,authorization_set_tuple_count,authorization_clear_tuple_count,authorization_recovered_count,authorization_state_upper_bound_authorities,authorization_tuple_gas
0,24120001,1,1,92,108,1,0,1,1,6912
1,24120002,3,3,276,324,3,0,3,3,20736
2,24120003,1,1,92,108,1,0,1,1,6912
3,24120004,1,1,92,108,1,0,1,1,6912
4,24120005,2,2,184,216,2,0,2,2,13824


## Pull Calldata And Blob Hash Counts

Calldata bytes come from Xatu. Blob versioned hashes are counted from Xatu `execution_transaction.blob_hashes`.

For EIP-8131 floor accounting, each blob versioned hash contributes `32 bytes * 64 gas/byte`.

In [4]:
calldata = query_xatu_calldata_by_block(raw_client, BLOCKS, network=NETWORK)

if WRITE_CSV:
    calldata.to_csv(CALLDATA_CSV, index=False)
    print(CALLDATA_CSV)

calldata[[
    "block_number",
    "calldata_zero_bytes",
    "calldata_nonzero_bytes",
    "calldata_bytes",
    "calldata_gas",
    "blob_versioned_hash_count",
    "blob_versioned_hash_bytes",
    "blob_versioned_hash_gas",
]].head()

/Users/william/PycharmProjects/eip-7999-research/data/xatu_calldata_24120001_24120050.csv


,block_number,calldata_zero_bytes,calldata_nonzero_bytes,calldata_bytes,calldata_gas,blob_versioned_hash_count,blob_versioned_hash_bytes,blob_versioned_hash_gas
0,24120001,144495,94030,238525,2082460,13,416,26624
1,24120002,29852,14640,44492,353648,6,192,12288
2,24120003,86377,50190,136567,1148548,4,128,8192
3,24120004,44058,26402,70460,598664,9,288,18432
4,24120005,30210,16786,46996,389416,0,0,0


## Load Existing BAL Estimates

This notebook expects the RPC BAL notebook to have already written `rpc_bal_summary_{start}_{end}.csv`. The join uses raw RLP BAL bytes with reads and system changes, matching the current BAL estimation path.

In [5]:
if not BAL_CSV.exists():
    raise FileNotFoundError(
        f"Missing {BAL_CSV}. Run notebooks/0.3-rpc-bal-rlp.ipynb for the same block range first."
    )

bal = pd.read_csv(BAL_CSV)
bal = bal[["block_number", "bal_rlp_bytes"]].copy()
bal["bal_gas"] = bal["bal_rlp_bytes"] * 16
bal.head()

,block_number,bal_rlp_bytes,bal_gas
0,24120001,339327,5429232
1,24120002,72309,1156944
2,24120003,186956,2991296
3,24120004,117839,1885424
4,24120005,81624,1305984


## Join Bandwidth Content

This table joins the bandwidth components currently available and keeps only bytes/gas fields plus the counts needed to understand fixed-size components.

```text
bandwidth_payload_bytes = calldata bytes + BAL RLP bytes + actual authorization RLP bytes + blob versioned hash bytes
bandwidth_metered_bytes = calldata bytes + BAL RLP bytes + EIP-8131 authorization bytes + blob versioned hash bytes
bandwidth_gas = calldata gas + BAL gas + authorization tuple gas + blob versioned hash gas
```

For authorization tuples, `authorization_tuple_rlp_bytes` is the actual encoded payload size, while `authorization_tuple_8131_bytes` is the EIP-8131 fixed-size byte count used for floor gas.

In [6]:
content = calldata.merge(bal, on="block_number", how="left")
content = content.merge(auth_summary, on="block_number", how="left")

fill_zero_cols = [
    "bal_rlp_bytes",
    "bal_gas",
    "type4_tx_count",
    "authorization_tuple_count",
    "authorization_tuple_rlp_bytes",
    "authorization_tuple_8131_bytes",
    "blob_versioned_hash_count",
    "blob_versioned_hash_bytes",
    "blob_versioned_hash_gas",
    "authorization_tuple_gas",
]
for column in fill_zero_cols:
    if column in content.columns:
        content[column] = content[column].fillna(0).astype("int64")

content["bandwidth_payload_bytes"] = (
    content["calldata_bytes"]
    + content["bal_rlp_bytes"]
    + content["authorization_tuple_rlp_bytes"]
    + content["blob_versioned_hash_bytes"]
)
content["bandwidth_metered_bytes"] = (
    content["calldata_bytes"]
    + content["bal_rlp_bytes"]
    + content["authorization_tuple_8131_bytes"]
    + content["blob_versioned_hash_bytes"]
)
content["bandwidth_gas"] = (
    content["calldata_gas"].astype("Int64")
    + content["bal_gas"].astype("Int64")
    + content["authorization_tuple_gas"].astype("Int64")
    + content["blob_versioned_hash_gas"].astype("Int64")
)

output_cols = [
    "block_number",
    "calldata_zero_bytes",
    "calldata_nonzero_bytes",
    "calldata_bytes",
    "calldata_gas",
    "bal_rlp_bytes",
    "bal_gas",
    "authorization_tuple_count",
    "authorization_tuple_rlp_bytes",
    "authorization_tuple_8131_bytes",
    "authorization_tuple_gas",
    "blob_versioned_hash_count",
    "blob_versioned_hash_bytes",
    "blob_versioned_hash_gas",
    "bandwidth_payload_bytes",
    "bandwidth_metered_bytes",
    "bandwidth_gas",
]
content = content[output_cols].sort_values("block_number").reset_index(drop=True)

if WRITE_CSV:
    content.to_csv(CONTENT_CSV, index=False)
    print(CONTENT_CSV)

content.head()

/Users/william/PycharmProjects/eip-7999-research/data/bandwidth_content_8131_24120001_24120050.csv


,block_number,calldata_zero_bytes,calldata_nonzero_bytes,calldata_bytes,calldata_gas,bal_rlp_bytes,bal_gas,authorization_tuple_count,authorization_tuple_rlp_bytes,authorization_tuple_8131_bytes,authorization_tuple_gas,blob_versioned_hash_count,blob_versioned_hash_bytes,blob_versioned_hash_gas,bandwidth_payload_bytes,bandwidth_metered_bytes,bandwidth_gas
0,24120001,144495,94030,238525,2082460,339327,5429232,1,92,108,6912,13,416,26624,578360,578376,7545228
1,24120002,29852,14640,44492,353648,72309,1156944,3,276,324,20736,6,192,12288,117269,117317,1543616
2,24120003,86377,50190,136567,1148548,186956,2991296,1,92,108,6912,4,128,8192,323743,323759,4154948
3,24120004,44058,26402,70460,598664,117839,1885424,1,92,108,6912,9,288,18432,188679,188695,2509432
4,24120005,30210,16786,46996,389416,81624,1305984,2,184,216,13824,0,0,0,128804,128836,1709224


## Summary

In [7]:
summary = pd.DataFrame(
    [
        {
            "blocks": len(content),
            "calldata_bytes": int(content["calldata_bytes"].sum()),
            "calldata_gas": int(content["calldata_gas"].sum()),
            "bal_rlp_bytes": int(content["bal_rlp_bytes"].sum()),
            "bal_gas": int(content["bal_gas"].sum()),
            "authorization_tuple_count": int(content["authorization_tuple_count"].sum()),
            "authorization_tuple_rlp_bytes": int(content["authorization_tuple_rlp_bytes"].sum()),
            "authorization_tuple_8131_bytes": int(content["authorization_tuple_8131_bytes"].sum()),
            "authorization_tuple_gas": int(content["authorization_tuple_gas"].sum()),
            "blob_versioned_hash_count": int(content["blob_versioned_hash_count"].sum()),
            "blob_versioned_hash_bytes": int(content["blob_versioned_hash_bytes"].sum()),
            "blob_versioned_hash_gas": int(content["blob_versioned_hash_gas"].sum()),
            "bandwidth_payload_bytes": int(content["bandwidth_payload_bytes"].sum()),
            "bandwidth_metered_bytes": int(content["bandwidth_metered_bytes"].sum()),
            "bandwidth_gas": int(content["bandwidth_gas"].sum()),
        }
    ]
)
summary

,blocks,calldata_bytes,calldata_gas,bal_rlp_bytes,bal_gas,authorization_tuple_count,authorization_tuple_rlp_bytes,authorization_tuple_8131_bytes,authorization_tuple_gas,blob_versioned_hash_count,blob_versioned_hash_bytes,blob_versioned_hash_gas,bandwidth_payload_bytes,bandwidth_metered_bytes,bandwidth_gas
0,50,5412521,46062116,7263438,116215008,67,6225,7236,463104,235,7520,481280,12689704,12690715,163221508
